# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NJ555/flyrank-ml-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
I used a classification model for this lane because the output is a 3-class action label: review_first, review_next, and monitor.

This is a tabular problem, so tree-based models fit well. I started with Logistic Regression as a simple baseline, then tried Random Forest and Gradient Boosting to see whether a stronger model improves the validation score.

I also kept the model easy to explain, because the final notebook should show not only score, but also what the model is learning from the data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
I used a grouped split by client_id. This is a fairer test because rows from the same client can look very similar, and I do not want the same client in both train and validation.

I kept the validation set untouched during training. This makes the result closer to a real unseen-client check and avoids leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Load the Week-4 scored file
data_path = Path("work/outputs/baseline_action_score.csv")
df = pd.read_csv(data_path)

target = "action_label"
group_col = "client_id"

# Basic safety checks
if target not in df.columns:
    raise ValueError("action_label not found. Use work/outputs/baseline_action_score.csv from Week 4.")
if group_col not in df.columns:
    raise ValueError("client_id not found in the file.")

# Drop target, group, and leakage / derived columns
drop_cols = [target, group_col]
for c in ["score", "rank", "reason_code", "content_id"]:
    if c in df.columns:
        drop_cols.append(c)

X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[target]
groups = df[group_col]

# Group split
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

# Column types
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ]), categorical_cols),
    ],
    remainder="drop"
)

models = {
    "Dummy most_frequent": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample"
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

rows = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)

    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_val, preds),
        "macro_f1": f1_score(y_val, preds, average="macro"),
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False)
results_df

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
The model is strongest on rows with clear signals and weaker on cases where the labels are close together.

The most common confusion is usually between review_next and monitor, because those rows often sit near the decision boundary. The model also depends most on the strongest observed features such as CTR, average position, impressions, clicks, and freshness-related columns.

This means the model is learning the same kind of pattern that the Week-4 rule used, but in a more flexible way.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
best_preds = best_model.predict(X_val)

print("Best model:", best_name)
print()
print(classification_report(y_val, best_preds))
print("Confusion matrix:")
print(confusion_matrix(y_val, best_preds))

# Feature importance for Random Forest
rf_model = fitted_models["Random Forest"]
feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
importances = rf_model.named_steps["model"].feature_importances_

feat_imp = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

feat_imp.head(15)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.